## Creating the infrastructure
### 1. Simulating transactions into a table with multiple records

In [0]:
%sql
-- Creating a Delta table as first
CREATE OR REPLACE TABLE scotiabank_transactions_alberta AS
WITH 
-- 1. Create a sequence with timestamps each 30 minutes for the last 3 days
time_series AS (
    SELECT EXPLODE(sequence(
        current_timestamp() - INTERVAL 3 DAY, 
        current_timestamp(), 
        INTERVAL 30 MINUTE
    )) AS transaction_timestamp
),

-- 2. Duplicate each interval 10 times to simulate 10 transactions per interval
expansion AS (
    SELECT t.transaction_timestamp
    FROM time_series t
    CROSS JOIN (SELECT EXPLODE(sequence(1, 10)) AS seq)
),

-- 3. Define the list of the reference catalogs
catalogs AS (
    SELECT 
        array('SCOTIA-CA-001', 'SCOTIA-CA-002', 'SCOTIA-CA-003', 'SCOTIA-CA-004', 'SCOTIA-CA-005', 
              'SCOTIA-CA-006', 'SCOTIA-CA-007', 'SCOTIA-CA-008', 'SCOTIA-CA-009', 'SCOTIA-CA-010') AS cards,
        array('Calgary', 'Edmonton', 'Red Deer', 'Banff', 'Lethbridge') AS cities,
        array('Supermarket', 'Gas Station', 'Restaurant', 'Electronics') AS merchants
)

-- 4. Selecting random values from the catalogs with 5% probability of anomaly detected
SELECT 
    e.transaction_timestamp,
    c.cards[CAST(rand() * 9 AS INT)] AS card_id,
    CASE 
        -- Inject anomaly with a random amount between 2500 and 5000
        WHEN rand() < 0.05 THEN ROUND(2500 + rand() * 2500, 2)
        -- Normal amount between 20 and 200
        ELSE ROUND(20 + rand() * 180, 2)
    END AS transaction_amount,
    c.merchants[CAST(rand() * 3 AS INT)] AS merchant_category,
    c.cities[CAST(rand() * 4 AS INT)] AS city
FROM expansion e
CROSS JOIN catalogs c;

### Checking the results

In [0]:
%sql
SELECT * FROM scotiabank_transactions_alberta LIMIT 10;

### 2. Split data into historical and incoming batches

In [0]:
%sql
-- 1. Historical data used to train the machine learning model (transactions older than 6 hours)
CREATE OR REPLACE TABLE scotiabank_training_data AS
SELECT * FROM scotiabank_transactions_alberta
WHERE transaction_timestamp <= (current_timestamp() - INTERVAL 6 HOUR);

-- 2. Recent incoming transactions (the last 6 hours where we want to detect anomalies)
CREATE OR REPLACE TABLE scotiabank_incoming_transactions AS
SELECT * FROM scotiabank_transactions_alberta
WHERE transaction_timestamp > (current_timestamp() - INTERVAL 6 HOUR);

-- Verify the row counts of both tables
SELECT 'Training Data' AS dataset, COUNT(*) AS total_rows FROM scotiabank_training_data
UNION ALL
SELECT 'Incoming Data' AS dataset, COUNT(*) AS total_rows FROM scotiabank_incoming_transactions;

### 3. Train and detect anomalies using Python and scikit-learn

In [0]:
import pandas as pd
from sklearn.ensemble import IsolationForest

# 1. Load training data from Delta Lake into a Pandas DataFrame
df_train = spark.read.table("scotiabank_training_data").toPandas()

# 2. Load incoming transactions data to evaluate
df_incoming = spark.read.table("scotiabank_incoming_transactions").toPandas()

# 3. Initialize and train the Isolation Forest model
# We set contamination=0.05 because we injected roughly 5% anomalies in our simulation
model = IsolationForest(contamination=0.05, random_state=42)

# Fit the model using the transaction amounts from the historical training data
model.fit(df_train[["transaction_amount"]])

# 4. Predict anomalies on the incoming transactions
# IsolationForest returns 1 for normal records and -1 for anomalies
df_incoming["raw_prediction"] = model.predict(df_incoming[["transaction_amount"]])

# Convert predictions to a boolean flag (True if anomaly, False otherwise)
df_incoming["is_anomaly"] = df_incoming["raw_prediction"] == -1

# Calculate an anomaly score (negative anomaly score where lower values indicate more severe anomalies)
df_incoming["anomaly_score"] = model.decision_function(df_incoming[["transaction_amount"]])

# 5. Filter only the detected anomalies and sort by severity (most severe/lowest score first)
detected_anomalies = df_incoming[df_incoming["is_anomaly"] == True].sort_values(
    by="anomaly_score", ascending=True
)

# 6. Save the results back into a Delta Lake table in Databricks
spark_anomalies_df = spark.createDataFrame(detected_anomalies)
spark_anomalies_df.write.mode("overwrite").saveAsTable("detected_anomalies_databricks")

# Display the top detected anomalies in the notebook output
display(detected_anomalies[[
    "transaction_timestamp", 
    "card_id", 
    "transaction_amount", 
    "merchant_category", 
    "city", 
    "anomaly_score"
]].head(10))

### 4. Checking the results

In [0]:
%sql
-- View the anomalies detected in Databricks, ordered from most serious to least serious
SELECT 
    transaction_timestamp,
    card_id,
    transaction_amount,
    merchant_category,
    city,
    anomaly_score, -- The most negative is this value the most serious anomaly
    is_anomaly
FROM 
    detected_anomalies_databricks
ORDER BY 
    anomaly_score ASC; -- Order from most serious to least serious